# 【完全自動・100%藤原紀香】専用AI音声モデル（Style-Bert-VITS2 JP-Extra）学習パイプライン

誤字・脱字を100%完全校正した「公式生声50クリップ」から、
**活舌明瞭・ネイティブ日本語アクセントの藤原紀香専用音声合成モデル** を完全自動・ワンクリックで学習します。

---
### 実行手順（わずか 1 クリック・ファイル選択不要）
1. 下のコードセルの再生ボタン（▶）をクリックするだけ。
※ 完全校正済みデータセット（20MB）が自動ダウンロードされ、T4 GPU で約15分でファインチューニングを完走します。
※ 完了すると、完成モデル (`norika_official_model.zip`) が自動でブラウザからダウンロードされます。

In [ ]:
# ==========================================
# 【完全校正版】藤原紀香 Style-Bert-VITS2 完全自動学習パイプライン
# ==========================================
from google.colab import files
import os, sys, shutil, zipfile, yaml, subprocess, urllib.request
from pathlib import Path

# 1. 完全校正済みデータセットの自動取得 (パブリック高速ミラーから完全自動ダウンロード・手動操作ゼロ)
%cd /content
zip_target = Path("/content/true_norika_colab_dataset.zip")
dataset_url = "https://raw.githubusercontent.com/moshajmoshaj/norika-voice-dataset/main/true_norika_colab_dataset.zip"

# 不完全・破損ファイル（15MB未満）が存在する場合は確実に削除
if zip_target.exists() and zip_target.stat().st_size < 15000000:
    print(f"🗑️ 破損・不完全な既存ファイル（{zip_target.stat().st_size} bytes）をクリーンアップ中...")
    zip_target.unlink()

if not zip_target.exists():
    print("📥 最新の完全校正済みデータセット（誤字撲滅版・20MB）を高速ダウンロード中...")
    urllib.request.urlretrieve(dataset_url, str(zip_target))

if not zip_target.exists() or zip_target.stat().st_size < 15000000:
    !curl -L -f -o /content/true_norika_colab_dataset.zip "https://raw.githubusercontent.com/moshajmoshaj/norika-voice-dataset/main/true_norika_colab_dataset.zip"

assert zip_target.exists() and zip_target.stat().st_size > 15000000, f"データセット取得失敗: {zip_target.stat().st_size if zip_target.exists() else 0} bytes"
print(f"✅ 完全校正済みデータセット確認完了！ ({zip_target.stat().st_size/1024/1024:.2f} MB)")

# 2. CMake 安定版 ＆ uv パッケージマネージャ導入
os.environ["PATH"] += ":/root/.cargo/bin"
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
!pip install -q "cmake<3.31"
if not os.path.exists("/usr/local/bin/uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh

# 3. Style-Bert-VITS2 公式基盤セットアップ
if not os.path.exists("/content/Style-Bert-VITS2"):
    !git clone https://github.com/litagin02/Style-Bert-VITS2.git /content/Style-Bert-VITS2
    %cd /content/Style-Bert-VITS2
    !CMAKE_POLICY_VERSION_MINIMUM=3.5 uv pip install --system -r requirements-colab.txt --no-progress
    !python initialize.py --skip_default_models
else:
    %cd /content/Style-Bert-VITS2

# 3.1 最新環境（PyTorch 2.6+ / TorchAudio 2.9+ / Numba / TensorBoard）完全互換パッチ
print("🔧 最新ライブラリ互換パッチを適用中...")
os.environ["TENSORBOARD_NO_TF"] = "1"
!pip uninstall -q -y tensorflow
!pip install -q "numpy==1.26.4"
!sed -i 's/-> torchaudio\.AudioMetaData:/-> object:/g' /usr/local/lib/python3*/dist-packages/pyannote/audio/core/io.py 2>/dev/null || true

# torchaudio クリーンパッチ (NameError 根絶)
try:
    import torchaudio
    ta_path = Path(torchaudio.__file__)
    ta_lines = [l for l in ta_path.read_text(encoding="utf-8").splitlines() if "hasattr(torchaudio" not in l and "AudioMetaData" not in l and "list_audio_backends" not in l]
    ta_patch = "\n\nclass AudioMetaData:\n    pass\n\ndef list_audio_backends():\n    return ['sox_io']\n"
    ta_path.write_text("\n".join(ta_lines) + ta_patch, encoding="utf-8")
    print("  ✅ torchaudio パッチ適用完了")
except Exception as e:
    print(f"  ⚠️ torchaudio note: {e}")

# style_gen.py / train_ms_jp_extra.py の torch.load パッチ (weights_only=False)
for script_name in ["style_gen.py", "train_ms_jp_extra.py"]:
    sp = Path(f"/content/Style-Bert-VITS2/{script_name}")
    if sp.exists():
        s_txt = sp.read_text(encoding="utf-8")
        if "_orig_load" not in s_txt:
            load_patch = "import torch\n_orig_load = torch.load\ntorch.load = lambda *a, **k: _orig_load(*a, **{**k, 'weights_only': False})\n"
            sp.write_text(s_txt.replace("import torch\n", load_patch, 1), encoding="utf-8")
print("  ✅ torch.load (weights_only=False) パッチ適用完了")

print("✅ 環境構築＆互換パッチ適用完了！")

# 4. データセットの自動展開と配置
!rm -rf /content/temp_dataset
with zipfile.ZipFile(str(zip_target), 'r') as z:
    z.extractall("/content/temp_dataset")

temp_dir = Path("/content/temp_dataset")
with open(temp_dir / "esd.list", "r", encoding="utf-8") as f:
    model_name = f.readline().strip().split("|")[1]
print(f"検出されたモデル名: {model_name}")

data_dir = Path(f"Data/{model_name}")
if data_dir.exists():
    shutil.rmtree(data_dir)
raw_dir = data_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

for f in (temp_dir / "wavs").glob("*.wav"):
    shutil.copy(f, raw_dir / f.name)
shutil.copy(temp_dir / "esd.list", data_dir / "esd.list")

with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": "/content/Style-Bert-VITS2/Data", "assets_root": "/content/Style-Bert-VITS2/model_assets"}, f)
print("✅ データセット配置完了！")

# 5. 日本語特化前処理 ＆ ファインチューニングの実行 (120 Epochs で滑舌・子音明瞭化)
pipeline_code = f'''
import yaml, subprocess
from gradio_tabs.train import preprocess_all, get_path
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker

model_name = "{model_name}"
print("🔄 日本語BERT特徴量抽出・スタイルベクトル生成中...")
pyopenjtalk_worker.initialize_worker()
preprocess_all(
    model_name=model_name,
    batch_size=4,
    epochs=120,
    save_every_steps=1000,
    num_processes=2,
    normalize=False,
    trim=False,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=True,
    val_per_lang=0,
    log_interval=200,
    yomi_error="skip",
)
print("✅ 前処理完了！")

paths = get_path(model_name)
with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

print("🚀 T4 GPU での日本語特化ファインチューニングを開始します（約15分）...")
cmd = [
    "python", "train_ms_jp_extra.py",
    "--config", str(paths.config_path),
    "--model", str(paths.dataset_path),
    "--assets_root", "/content/Style-Bert-VITS2/model_assets"
]
subprocess.run(cmd, check=True)
print("🎉 学習完了！")
'''

with open("run_pipeline.py", "w", encoding="utf-8") as f:
    f.write(pipeline_code)

!python run_pipeline.py

# 6. 完成モデルの自動ダウンロード
print("📦 完成モデル（norika_official_model.zip）をパッケージング中...")
!zip -r /content/norika_official_model.zip model_assets/
print("⬇️ ブラウザからダウンロードを開始します...")
files.download("/content/norika_official_model.zip")
print("※ もし自動ダウンロードが始まらない場合は、左側のファイル一覧（📁アイコン）から norika_official_model.zip を右クリックして『ダウンロード』してください。")
print("✨ すべて完了しました！ダウンロードされた zip を手元 PC の models/norika_vits/ に解凍配置してください。")
